# One-Hot Encoding

One-hot encoding converts categorical values into separate binary (0/1) columns. It is useful when categories do not have a meaningful order, such as fuel type or previous owner category.

For example, a `fuel` column containing `Petrol`, `Diesel`, and `CNG` can become three indicator columns. A row receives `1` in the column matching its category and `0` in the others.

This notebook covers three practical approaches:

1. One-hot encoding with **Pandas** (`pd.get_dummies()`).
2. One-hot encoding with **Scikit-learn** (`OneHotEncoder`), fitting on training data and transforming test data.
3. Grouping infrequent categories into a single `uncommon` category before encoding.

In [1]:
import numpy as np 
import pandas as pd 

In [2]:
df = pd.read_csv('Datasets/cars.csv')

In [3]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [4]:
df['brand'].nunique()

32

#### One-hot encoding with Pandas

`pd.get_dummies()` converts the selected categorical columns (`fuel` and `owner`) into indicator columns.

- `columns`: selects which columns should be encoded.
- `dtype=int`: stores the dummy values as integers (`0` and `1`).

The original selected columns are replaced by their generated dummy columns. Other columns remain in the DataFrame.

In [5]:
pd.get_dummies(df, columns=['fuel', 'owner'], dtype=int)

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,0,1,1,0,0,0,0
8124,Hyundai,119000,135000,0,1,0,0,0,1,0,0,0
8125,Maruti,120000,382000,0,1,0,0,1,0,0,0,0
8126,Tata,25000,290000,0,1,0,0,1,0,0,0,0


**Drop the first dummy column**

`drop_first=True` removes one category column from each encoded feature. For a feature with \(k\) categories, this leaves \(k-1\) columns.

This can avoid the *dummy variable trap* (perfect linear dependence among the full set of dummy columns) in models that include an intercept, such as ordinary linear regression.

The dropped category becomes the reference category: when all remaining dummy columns for that feature are zero, the row belongs to that reference category. Dropping a column is not automatically necessary for every model; tree-based models, for example, generally do not need it.

In [6]:
# delete 1 cols from ervery dummy cols to remove multicolinearity issue 
pd.get_dummies(df, columns=['fuel', 'owner'], dtype=int, drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


### Using Sklearn 

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 0:4], df.iloc[:, -1], test_size=0.2, random_state=0)

In [8]:
X_train

,brand,km_driven,fuel,owner
3042,Hyundai,60000,LPG,First Owner
1520,Tata,150000,Diesel,Third Owner
2611,Hyundai,110000,Diesel,Second Owner
3544,Mahindra,28000,Diesel,Second Owner
4138,Maruti,15000,Petrol,First Owner
...,...,...,...,...
4931,Tata,70000,Diesel,Third Owner
3264,Ford,100000,Diesel,Second Owner
1653,Hyundai,90000,Petrol,Second Owner
2607,Volkswagen,90000,Diesel,First Owner


In [9]:
y_train

3042     375000
1520     150000
2611     459999
3544    1050000
4138     700000
         ...   
4931     300000
3264     220000
1653     170000
2607     375000
2732     225000
Name: selling_price, Length: 6502, dtype: int64

In [10]:
from sklearn.preprocessing import OneHotEncoder

In [11]:
ohe = OneHotEncoder(drop='first', dtype=np.int32)

In [12]:
X_train_new = ohe.fit_transform(X_train[['fuel', 'owner']]).toarray()

In [13]:
X_test_new = ohe.transform(X_test[['fuel', 'owner']]).toarray()

In [14]:
X_train[['brand', 'km_driven']].values

array([['Hyundai', 60000],
       ['Tata', 150000],
       ['Hyundai', 110000],
       ...,
       ['Hyundai', 90000],
       ['Volkswagen', 90000],
       ['Hyundai', 110000]], dtype=object)

In [15]:
np.hstack((X_train[['brand', 'km_driven']].values, X_train_new)) 

array([['Hyundai', 60000, 0, ..., 0, 0, 0],
       ['Tata', 150000, 1, ..., 0, 0, 1],
       ['Hyundai', 110000, 1, ..., 1, 0, 0],
       ...,
       ['Hyundai', 90000, 0, ..., 1, 0, 0],
       ['Volkswagen', 90000, 1, ..., 0, 0, 0],
       ['Hyundai', 110000, 0, ..., 0, 0, 0]], dtype=object)

#### Encode the Top Categories 

In [16]:
# here in brand we have 32 caregories 
df['brand'].nunique()

32

In [17]:
counts = df['brand'].value_counts()
counts

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [18]:
threshold = 100 

In [19]:
# here we filter where number of cars in less than 100 
repl = counts[counts <= threshold].index
repl

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Land', 'Force', 'Isuzu', 'Ambassador',
       'Kia', 'MG', 'Daewoo', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [20]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon'), dtype=np.int16).sample(7)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
2796,0,0,0,0,1,0,0,0,0,0,0,0,0
1456,0,0,0,1,0,0,0,0,0,0,0,0,0
5127,0,0,0,0,0,0,1,0,0,0,0,0,0
4585,0,0,0,0,0,0,1,0,0,0,0,0,0
7202,0,0,0,0,1,0,0,0,0,0,0,0,0
3596,0,0,0,0,1,0,0,0,0,0,0,0,0
160,0,0,0,0,0,0,0,0,0,0,1,0,0


### Advantages and limitations

**Advantages**
- Converts nominal categories into numeric indicator columns without assigning an artificial ranking.
- Works with many common classification and regression algorithms.
- Easy to implement with Pandas or Scikit-learn.
- Grouping rare categories can reduce the number of generated features.

**Limitations**
- Can create many columns when a feature has high cardinality.
- The resulting feature matrix can use substantial memory, especially when converted to a dense array.
- Categories not seen during fitting need to be handled explicitly by the encoder.
- One-hot encoding does not capture similarity or relationships between category names.

### Pandas vs Scikit-learn

| Approach | Best suited for | Important consideration |
|---|---|---|
| `pd.get_dummies()` | Quick encoding and data exploration | Align train/test columns carefully if encoding them separately |
| `OneHotEncoder` | ML preprocessing workflows | Fit on training data and reuse the fitted encoder for test/new data |

### Key takeaways

1. One-hot encoding is suitable for nominal categories that have no natural order.
2. Use `drop_first=True` in Pandas or `drop='first'` in Scikit-learn when a \(k-1\) representation is useful, particularly for linear models with an intercept.
3. Split the data first, fit the encoder on training data, and transform the test data with the same encoder.
4. Consider sparse output for high-cardinality features to reduce memory use.
5. Group rare categories when it is justified by the category distribution and the modelling task.
6. In a production workflow, keep the fitted encoder and apply the same preprocessing to incoming data.